# 00b — Collect `cladding_panel` images from Wikimedia Commons

The first collection run returned almost no real cladding facades, because a plain keyword search matches any file that mentions "cladding".
This notebook targets modern metal / aluminium / composite-panel facades and keeps only freely licensed images.

**Run order:** cell 1 (settings) → cell 2 (download) → cell 3 (preview and delete bad images) → cell 4 (zip and download).

Output: `/content/raw_images/cladding_panel_v2/` with the images and `sources.csv` (title, license, author, source page) for the README attribution.

In [ ]:
# Cell 1 — settings
from pathlib import Path

TARGET = 50            # images to keep
MIN_WIDTH = 800        # skip small images
OUT_DIR = Path("/content/raw_images/cladding_panel_v2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Licenses safe to redistribute in the GitHub Release (attribution goes in sources.csv)
ALLOWED_LICENSES = ("cc0", "public domain", "pd", "cc by", "cc-by")

# Words that usually mean the file is not a facade photo
EXCLUDE = " -interior -diagram -drawing -logo -map -plan -section -detail -sample -texture"

# Category searches first (most precise; a category that doesn't exist just returns nothing),
# then targeted keyword searches.
QUERIES = [
    'incategory:"Aluminium facades"',
    'incategory:"Metal facades"',
    'incategory:"Metal cladding"',
    'incategory:"Aluminium composite panels"',
    'incategory:"Rainscreen cladding"',
    'incategory:"Metal-clad buildings"',
    '"aluminium cladding" building' + EXCLUDE,
    '"aluminum cladding" building' + EXCLUDE,
    '"metal cladding" building' + EXCLUDE,
    '"composite panel" facade' + EXCLUDE,
    '"metal facade" modern building' + EXCLUDE,
    '"perforated metal" facade' + EXCLUDE,
    '"corrugated metal" facade building' + EXCLUDE,
    '"zinc cladding" building' + EXCLUDE,
    '"metal panels" facade office building' + EXCLUDE,
]
print(f"{len(QUERIES)} queries, target {TARGET} images -> {OUT_DIR}")


In [ ]:
# Cell 2 — search and download
import requests, re, csv, time, html

API = "https://commons.wikimedia.org/w/api.php"
session = requests.Session()
# Wikimedia asks every script to identify itself
session.headers["User-Agent"] = "M4U3-facade-materials-dataset/1.0 (student project; contact via GitHub)"

def clean(text):
    return html.unescape(re.sub(r"<[^>]+>", "", text or "")).strip()

def search(query, limit=50):
    params = {
        "action": "query", "format": "json",
        "generator": "search", "gsrsearch": query + " filetype:bitmap",
        "gsrnamespace": 6, "gsrlimit": limit,
        "prop": "imageinfo", "iiprop": "url|size|extmetadata", "iiurlwidth": 1024,
    }
    r = session.get(API, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("query", {}).get("pages", {}).values()

seen, rows = set(), []
for q in QUERIES:
    if len(rows) >= TARGET:
        break
    try:
        pages = list(search(q))
    except Exception as e:
        print(f"  query failed: {q} ({e})")
        continue
    kept_here = 0
    for p in pages:
        if len(rows) >= TARGET:
            break
        title = p.get("title", "")
        if title in seen or not p.get("imageinfo"):
            continue
        seen.add(title)
        info = p["imageinfo"][0]
        meta = info.get("extmetadata", {})
        lic = clean(meta.get("LicenseShortName", {}).get("value", "")).lower()
        if not lic.startswith(ALLOWED_LICENSES) or info.get("width", 0) < MIN_WIDTH:
            continue
        url = info.get("thumburl") or info.get("url")
        fname = f"cladding_panel_{len(rows):03d}.jpg"
        try:
            img = session.get(url, timeout=30)
            img.raise_for_status()
            (OUT_DIR / fname).write_bytes(img.content)
        except Exception as e:
            print(f"  download failed: {title} ({e})")
            continue
        rows.append({
            "file": fname,
            "title": title,
            "license": clean(meta.get("LicenseShortName", {}).get("value", "")),
            "author": clean(meta.get("Artist", {}).get("value", "unknown"))[:120],
            "source_page": info.get("descriptionurl", ""),
            "query": q,
        })
        kept_here += 1
        time.sleep(0.2)   # be polite to the API
    print(f"{kept_here:3d} kept  <-  {q}")

with open(OUT_DIR / "sources.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file", "title", "license", "author", "source_page", "query"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nSaved {len(rows)} images and sources.csv to {OUT_DIR}")


In [ ]:
# Cell 3 — preview: look at every image and delete the ones that are not cladding
import matplotlib.pyplot as plt
from PIL import Image

files = sorted(OUT_DIR.glob("*.jpg"))
assert files, "No images found - run cell 2 first"
cols = 5
rows_n = (len(files) + cols - 1) // cols
fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 3.2, rows_n * 3.2))
for ax in axes.flat:
    ax.axis("off")
for ax, f in zip(axes.flat, files):
    ax.imshow(Image.open(f).convert("RGB"))
    ax.set_title(f.stem.split("_")[-1], fontsize=10)
plt.tight_layout()
plt.show()

# Put the numbers of the bad images here and re-run this cell, e.g. BAD = [3, 17, 22]
BAD = []
for n in BAD:
    p = OUT_DIR / f"cladding_panel_{n:03d}.jpg"
    if p.exists():
        p.unlink()
        print("deleted", p.name)
print(len(list(OUT_DIR.glob("*.jpg"))), "images remain")


In [ ]:
# Cell 4 — drop deleted images from sources.csv, then zip and download
import csv, shutil
from google.colab import files as colab_files

src_csv = OUT_DIR / "sources.csv"
with open(src_csv, encoding="utf-8") as f:
    kept = [r for r in csv.DictReader(f) if (OUT_DIR / r["file"]).exists()]
assert kept, "No images left to zip"
with open(src_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(kept[0].keys()))
    writer.writeheader()
    writer.writerows(kept)
print(len(kept), "images listed in sources.csv")

shutil.make_archive("/content/cladding_panel_v2", "zip", OUT_DIR)
colab_files.download("/content/cladding_panel_v2.zip")
